# Project 2 — Notebook 04: Customer Lifetime Value (CLTV)

**Week 3 goal:** calculate simple historical CLTV and compare customer segments.

This is historical CLTV, not a complex prediction model. It answers: “How much revenue did customers actually generate in the available data?”

## Step 1 — Import and load the cohort dataset

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)

import pandas as pd
import numpy as np

from src.data_prep import load_cohort_dataset, add_customer_segments

pd.set_option('display.max_columns', 30)

In [ ]:
df = load_cohort_dataset(DATA_DIR)
print('rows:', f'{len(df):,}')
print('customers:', f'{df["customer_id"].nunique():,}')
print('revenue: £', f'{df["line_total"].sum():,.2f}')

## Step 2 — Create one row per order

The raw data is line-item level, so one invoice can appear on many rows. AOV should be based on orders/invoices, not individual product lines.

In [ ]:
orders = (df.groupby('invoice_no')
          .agg(customer_id=('customer_id', 'first'),
               invoice_datetime=('invoice_datetime', 'min'),
               country=('country', 'first'),
               order_value=('line_total', 'sum'))
          .reset_index())

orders.head()

## Step 3 — Overall AOV, purchase frequency, and historical CLTV

In [ ]:
total_revenue = df['line_total'].sum()
unique_customers = df['customer_id'].nunique()
number_of_orders = orders['invoice_no'].nunique()

average_order_value = total_revenue / number_of_orders
purchase_frequency = number_of_orders / unique_customers
historical_cltv = total_revenue / unique_customers

summary = pd.DataFrame({
    'metric': ['Total revenue', 'Number of orders', 'Unique customers', 'Average order value', 'Purchase frequency', 'Historical CLTV'],
    'value': [total_revenue, number_of_orders, unique_customers, average_order_value, purchase_frequency, historical_cltv]
})
summary

## Step 4 — CLTV by country

Country is the best available business segment in this dataset. I show countries with at least 5 customers so one very small country does not distract too much.

In [ ]:
country_cltv = (df.groupby('country')
                .agg(customers=('customer_id', 'nunique'),
                     orders=('invoice_no', 'nunique'),
                     revenue=('line_total', 'sum'))
                .reset_index())

country_cltv['aov'] = country_cltv['revenue'] / country_cltv['orders']
country_cltv['purchase_frequency'] = country_cltv['orders'] / country_cltv['customers']
country_cltv['historical_cltv'] = country_cltv['revenue'] / country_cltv['customers']

country_cltv_filtered = country_cltv[country_cltv['customers'] >= 5].sort_values('historical_cltv', ascending=False)
country_cltv_filtered.head(10)

## Step 5 — Simple customer value segments

To keep the segmentation beginner-friendly, I group customers by their total spend:
- Low value: bottom 50%
- Medium value: 50% to 80%
- High value: top 20%

In [ ]:
segmented = add_customer_segments(df)

segment_summary = (segmented.groupby('value_segment')
                   .agg(customers=('customer_id', 'nunique'),
                        orders=('invoice_no', 'nunique'),
                        revenue=('line_total', 'sum'))
                   .reset_index())

segment_summary['aov'] = segment_summary['revenue'] / segment_summary['orders']
segment_summary['purchase_frequency'] = segment_summary['orders'] / segment_summary['customers']
segment_summary['historical_cltv'] = segment_summary['revenue'] / segment_summary['customers']
segment_summary['revenue_share_percent'] = segment_summary['revenue'] / segment_summary['revenue'].sum() * 100
segment_summary.sort_values('historical_cltv', ascending=False)

## Step 6 — CLTV by acquisition cohort

In [ ]:
cohort_cltv = (df.groupby('cohort_month')
               .agg(customers=('customer_id', 'nunique'),
                    orders=('invoice_no', 'nunique'),
                    revenue=('line_total', 'sum'))
               .reset_index())
cohort_cltv['historical_cltv'] = cohort_cltv['revenue'] / cohort_cltv['customers']
cohort_cltv

## Step 7 — Save CLTV outputs

In [ ]:
summary.to_csv(OUTPUT_DIR / 'overall_cltv_summary.csv', index=False)
country_cltv.to_csv(OUTPUT_DIR / 'country_cltv_summary.csv', index=False)
segment_summary.to_csv(OUTPUT_DIR / 'customer_segment_summary.csv', index=False)
cohort_cltv.to_csv(OUTPUT_DIR / 'cohort_cltv_summary.csv', index=False)

print('Saved CLTV tables in:', OUTPUT_DIR)

## Week 3 notes

The high-value segment is small but produces most of the revenue. This tells me retention work should not treat every customer exactly the same. The business should protect high-value repeat buyers while also trying to move medium-value customers upward.